# 第 01 课 · TD(0) 小实验

只使用 Python 标准库，无需安装包，也无需 GPU。请先完成配套讲义的手算，再运行。

固定过程：A → B → T。奖励分别是 0 和 1，T 为终止状态。我们估计固定策略的状态价值，没有学习选择动作。

先写下预测：从全零价值开始，第一个回合中 A、B 哪一个会变化？为什么？
配套材料：[讲义](README.md)、[习题](exercises.md)、[参考解答](answers.md)。


In [ ]:
def td0_update(value, reward, next_value, gamma=0.9, alpha=0.1, terminal=False):
    target = reward if terminal else reward + gamma * next_value
    delta = target - value
    return value + alpha * delta, target, delta


a_after, a_target, a_delta = td0_update(0.2, 0.0, 0.5)
b_after, b_target, b_delta = td0_update(0.5, 1.0, 0.0, terminal=True)
print(f'A: target={a_target:.3f}, delta={a_delta:.3f}, new value={a_after:.3f}')
print(f'B: target={b_target:.3f}, delta={b_delta:.3f}, new value={b_after:.3f}')

## 终止状态：故意制造一个错误输入

假设其他代码把终点的价值误填为 100，正确的终止转移更新是否应该受到影响？先预测，再运行。这里的 terminal 指任务真正终止；后续学习环境接口时还会区分时间上限截断。

In [ ]:
correct, target, delta = td0_update(0.5, 1.0, 100.0, terminal=True)
incorrect, wrong_target, _ = td0_update(0.5, 1.0, 100.0, terminal=False)
print(f'正确识别终止状态: target={target:.3f}, new value={correct:.3f}')
print(f'误把它当作非终止: target={wrong_target:.3f}, new value={incorrect:.3f}')

## 训练固定策略的状态价值

每回合按 A → B → T 的顺序更新。更新 A 时，使用的是当时的 B 估计；不会提前使用这个回合稍后才更新的 B。

这里直接使用已知路径来生成经历，学习规则本身只看到逐步转移。最终真值只用于检查误差。

In [ ]:
def train_td(alpha=0.2, gamma=0.9, episodes=50):
    values = {'A': 0.0, 'B': 0.0}
    history = []
    for episode in range(1, episodes + 1):
        values['A'], _, _ = td0_update(
            values['A'], 0.0, values['B'], gamma, alpha
        )
        values['B'], _, _ = td0_update(
            values['B'], 1.0, 0.0, gamma, alpha, terminal=True
        )
        error = max(abs(values['A'] - gamma), abs(values['B'] - 1.0))
        history.append({'episode': episode, **values, 'max_error': error})
    return values, history


values, history = train_td()
print('回合       V(A)       V(B)    最大绝对误差')
for row in history[:5] + history[-2:]:
    print(f"{row['episode']:4d} {row['A']:10.5f} {row['B']:10.5f} {row['max_error']:14.5f}")
print('检查用真值:', {'A': 0.9, 'B': 1.0})

## 实验一：改变学习率

先预测 α = 0.05、0.2、0.8 哪个在本例中更快接近真值，再运行。

统一标准：两个状态的最大绝对误差首次小于 0.01 时的回合数。训练预算固定为 200 回合。未达到阈值也属于结果，不能删掉。

In [ ]:
def first_episode_below(history, tolerance=0.01):
    return next((row['episode'] for row in history if row['max_error'] < tolerance), None)


for learning_rate in (0.05, 0.2, 0.8):
    final_values, run_history = train_td(alpha=learning_rate, episodes=200)
    reached = first_episode_below(run_history)
    print(f'alpha={learning_rate:.2f}: 首次达标回合={reached}, 最终价值={final_values}')

请用两三句话解释结果。这个任务是确定性的：较快收敛的观察不能直接证明在随机奖励或神经网络训练中也应使用更大的学习率。

## 实验二：改变折扣因子

把下面的 gamma 改为 0.5。运行前先写下 A 和 B 的真值，以及为什么会变化。注意改变 gamma 也改变了衡量价值的方式。

In [ ]:
gamma_to_try = 0.9  # 作业：改为 0.5，先预测，再运行。
final_values, run_history = train_td(gamma=gamma_to_try, episodes=200)
print('gamma:', gamma_to_try)
print('学到的价值:', final_values)
print('检查用真值:', {'A': gamma_to_try, 'B': 1.0})

## 实验三：自己扩展路径

路径改成 A → B → C → T，只有 C → T 的奖励是 1，其余为 0。gamma = 0.9，alpha = 0.2，初始价值均为 0。

任务：

1. 手算每个状态的真值。
2. 使用 td0_update 写出训练循环，按实际转移顺序更新。
3. 在运行前预测第一、第二回合中哪些状态会变化。
4. 输出第 1、2、3、50 回合后的估计。

这部分故意留给你完成。直接运行下面的空白单元不会报错，但也不会替你完成实验。

In [ ]:
# 在这里写你的三状态实验。

## 提交与讨论

在学习记录中填写：

- 你运行前的预测和两次手算。
- 三个学习率的达标回合数，以及结论的适用范围。
- 改变 gamma 后的解释。
- 三状态扩展代码或关键输出。
- 讲义小测答案和一个仍然不确定的问题。

把“我预测什么—结果是什么—为什么”写清楚，比单独贴一段正确代码更重要。

原始学习资料：[赵世钰教材仓库](https://github.com/MathFoundationRL/Book-Mathematical-Foundation-of-Reinforcement-Learning)，第 7 章 Temporal-Difference Methods。